In [ ]:

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download resources once
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')


# Load your CSV
df = pd.read_csv('~/documents/document_references_fhir_NLP.csv')

# Clean text
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stopwords.words('english')]
    return ' '.join(tokens)

df['cleaned_text'] = df['note_text'].apply(clean_text)

df.head()

ckd_terms = [
    'chronic kidney disease', 
    'ckd', 
    'renal failure', 
    'renal insufficiency', 
    'end stage renal disease', 
    'esrd', 
    'elevated creatinine'
]

pattern = '|'.join(ckd_terms)

df['ckd_flag'] = df['cleaned_text'].str.contains(pattern, case=False, regex=True)

ckd_notes = df[df['ckd_flag']]
print(f"Found {len(ckd_notes)} CKD-related notes.")

symptoms = ['fatigue', 'swelling', 'edema', 'shortness of breath', 'proteinuria', 'creatinine']
df['symptom_flag'] = df['cleaned_text'].str.contains('|'.join(symptoms), case=False)

#symptoms = ['fatigue', 'swelling', 'edema', 'shortness of breath', 'proteinuria', 'creatinine']
#df['symptom_flag'] = df['cleaned_text'].str.contains('|'.join(symptoms), case=False)

ckd_notes.to_csv('ckd_notes.csv', index=False)






In [12]:
import spacy

#nlp = spacy.load("en_core_sci_sm")
nlp = spacy.load("en_core_sci_dm")


text = "Patient has chronic kidney disease, eGFR 45, and complains of fatigue."
doc = nlp(text)

for ent in doc.ents:
    print(ent.text, ent.label_)

df['entities'] = df['note_text'].apply(lambda x: [(ent.text, ent.label_) for ent in nlp(str(x))])

def has_ckd_and_symptoms(entities):
    diseases = [e[0].lower() for e in entities if e[1] == 'DISEASE']
    symptoms = [e[0].lower() for e in entities if e[1] == 'SYMPTOM']
    return 'chronic kidney disease' in diseases or 'ckd' in diseases

df['ckd_flag'] = df['entities'].apply(has_ckd_and_symptoms)
filtered = df[df['ckd_flag']]

import re

def extract_egfr(text):
    match = re.search(r'egrf\s*(\d+)', text.lower())
    return int(match.group(1)) if match else None

filtered['egfr'] = filtered['note_text'].apply(extract_egfr)





ModuleNotFoundError: No module named 'spacy'

In [5]:
import sys
print(sys.executable)

!{sys.executable} -m pip install spacy scispacy


/opt/anaconda3/envs/my_env/bin/python
  Using cached spacy-3.8.7-cp313-cp313-macosx_11_0_arm64.whl.metadata (27 kB)
  Using cached scispacy-0.5.5-py3-none-any.whl.metadata (18 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.13-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.2 kB)
  Using cached cymem-2.0.11-cp313-cp313-macosx_11_0_arm64.whl.metadata (8.5 kB)
  Using cached preshed-3.0.10-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached thinc-8.3.6-cp313-cp313-macosx_11_0_arm64.whl.metadata (15 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.5.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (19 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-0.4.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached typer-0.20.0-py3-none-any.whl.metadata (16 kB)
  Using cached tqdm-4